# 08 · Confirmatory small-context run, and seeds 2-4 for the W1 policies

**Part A: confirmatory run.** In the uniform runs (notebooks 03 and 06) the foundation models were ahead of XGBoost trained on the same rows in all 18 cells with a context of 1-2K rows. I noticed that *after* looking at the results, so it can't be claimed from the same runs. Here the same comparison is made on **new context draws** (`uniform_fresh`: same rule as uniform, a random order no earlier run used), with the hypothesis and the decision rule fixed beforehand in `docs/PROJECT_LOG.md` (25 Nov):

> With a context of at most 2,000 rows, TabPFN v2 and TabICLv2 reach higher PR-AUC than XGBoost trained on the same rows, and the gap is largest at the natural fraud rate.

Confirmed if the TFM is ahead in at least 13 of the 18 cells (one-sided sign test, 5%) and the mean gap at 0.172% is larger than at the other rates. The splits are the same 5 as everywhere else; only the contexts are new.

**Part B: seeds 2-4** for the policies that had only 2-3 seeds (TabICLv2 ensembles and kNN, and XGBoost on the stratified and balanced rows), so every policy in the W1 table has 5 seeds.

**Runtime (T4 GPU):** Part A about 1 hour (TabPFN v2 is most of it), Part B about 1-1.5 hours. Every run is saved to Drive as it finishes and skipped on a rerun, so a disconnect only costs the run in progress.

In [ ]:
# 1. What to run in this session.
SEEDS = [0, 1, 2, 3, 4]
RUN_PART_A = True            # confirmatory small-context run
RUN_PART_B = True            # seeds 2-4 for ensembles, kNN and matched XGBoost
LEVELS_ALL = ["20", "5", "1", "0.5", "natural"]
SMALL_K = [1_000, 2_000]     # the contexts the hypothesis is about
# Part B was cut to what the write-up needs after the first session showed one
# ensemble run takes ~6 minutes at 1K and much longer at 10K: the ensemble of
# balanced contexts (the best policy, compared with XGBoost at 0.172%) and XGBoost
# on the same balanced/stratified rows. Ensemble of uniform contexts and kNN keep
# their 2-3 seeds; the kNN failure is explained by notebook 09 instead.
PART_B_TFM_POLICIES = ["ensemble_balanced"]
PART_B_LEVELS = ["natural", "1"]

In [ ]:
# 2. Drive, latest code, libraries. Needs a GPU runtime (Runtime > Change runtime type > T4 GPU).
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4")

In [ ]:
# 3. Tests, data, and the split fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
from fraudicl.config import DRIVE_DIR, CONTEXT_BUDGETS
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
OUT = f"{DRIVE_DIR}/results"
print("splits OK", expected)

In [ ]:
# 4. Part A: the confirmatory runs. TabICLv2 first (fast), then XGBoost on the same
#    fresh rows, then TabPFN v2 (slowest), so a cut-off session still leaves complete
#    pairs for one foundation model. XGBoost is tuned on seed 0's fresh context for
#    each (fraud rate, k), the same scheme as the original matched runs.
from fraudicl.models import make_tabpfn_v2, make_tabicl_v2
from fraudicl.experiments import run_tfm_sweep, run_matched_xgb
if RUN_PART_A:
    run_tfm_sweep(X, y, splits, OUT, {"TabICLv2": make_tabicl_v2}, levels=LEVELS_ALL,
                  budgets=SMALL_K, seeds=SEEDS, policy="uniform_fresh")
    run_matched_xgb(X, y, splits, OUT, levels=LEVELS_ALL, budgets=SMALL_K, seeds=SEEDS,
                    policy="uniform_fresh", variant="plain", n_trials=50)
    run_tfm_sweep(X, y, splits, OUT, {"TabPFN v2": make_tabpfn_v2}, levels=LEVELS_ALL,
                  budgets=SMALL_K, seeds=SEEDS, policy="uniform_fresh")

In [ ]:
# 5. Part A: the test, exactly as fixed beforehand.
#    First: no fresh context may contain a validation or test row.
from fraudicl import report as R
from fraudicl.checks import context_leakage, matched_tests, confirmatory_verdict
leak = context_leakage(OUT, splits)
fresh = leak[leak.policy == "uniform_fresh"]
assert len(fresh) and (fresh[["outside_train", "in_validation", "in_test"]] == 0).all().all()
print(f"{len(fresh)} fresh contexts checked, none contains a validation or test row")

runs = R.load_runs(OUT)
confirm = matched_tests(OUT, y, runs, policy="uniform_fresh", n_boot=1000)
confirm = confirm[confirm.k <= 2_000]
verdict = confirmatory_verdict(confirm)
print(verdict)
confirm.round(4)

**How to read Part A.** One row per (model, fraud rate, k): `mean_diff` is TFM minus XGBoost on the same fresh rows, averaged over the 5 seeds; `seeds_tfm_ahead` out of 5; `ci_low`/`ci_high` and `p_value` from the pooled paired bootstrap; `p_holm` after Holm correction over the 18 cells.

The verdict uses only the rule fixed beforehand: `part1_direction_confirmed` (TFM ahead in at least 13 of 18 cells) and `part2_largest_at_natural`. Whatever it says is the result, including a "no": the exploratory finding then gets reported as not replicated.

In [ ]:
# 6. Part B: seeds 2-4 for the W1 policies that had fewer seeds. Ensembles first,
#    because the ensemble of balanced contexts is the best policy so far and its
#    comparison with XGBoost (0.858 vs 0.865) rests on 2 seeds.
from fraudicl.experiments import run_policy_sweep
if RUN_PART_B:
    # XGBoost on the same stratified and balanced rows first (balanced = random
    # undersampling): minutes, not hours, so it is never the part that gets cut off
    for policy in ["stratified", "balanced"]:
        run_matched_xgb(X, y, splits, OUT, levels=PART_B_LEVELS, budgets=CONTEXT_BUDGETS,
                        seeds=SEEDS, policy=policy, variant="plain", n_trials=50)
    run_policy_sweep(X, y, splits, OUT, {"TabICLv2": make_tabicl_v2}, PART_B_TFM_POLICIES,
                     levels=PART_B_LEVELS, budgets=CONTEXT_BUDGETS, seeds=SEEDS,
                     m_ensemble=5, ensemble_budgets=[1_000, 10_000])

In [ ]:
# 7. Part B: the W1 table again, now with every seed that has finished.
runs = R.load_runs(OUT)
policies = R.policy_table(runs, "TabICLv2", levels=["natural", "1"])
policies.round(4)

In [ ]:
# 8. Export: print the tables as CSV text (copied into results/ on the laptop),
#    and also save them on Drive.
import json
EXPORT = f"{OUT}/export_08"
os.makedirs(EXPORT, exist_ok=True)
confirm.to_csv(f"{EXPORT}/confirmatory_small_context.csv", index=False)
json.dump(verdict, open(f"{EXPORT}/confirmatory_verdict.json", "w"), indent=2)
policies.to_csv(f"{EXPORT}/policy_table_tabiclv2.csv", index=False)
print("=====CONFIRM====="); print(confirm.to_csv(index=False))
print("=====VERDICT====="); print(json.dumps(verdict))
print("=====POLICIES====="); print(policies.to_csv(index=False))